In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("bhavikjikadara/fake-news-detection")

print("Path to dataset files:", path)

In [ ]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', None)

In [ ]:
tweets = [
    "I love this phone, the battery life is amazing!",
    "The phone battery died after two hours. Very bad.",
    "Apple launched a new iPhone today.",
    "I ate an apple after lunch.",
    "The movie was amazing and the acting was excellent.",
    "This movie is terrible, I want my money back.",
    "Python is great for machine learning.",
    "I am learning Python programming for my project.",
    "The bank approved my loan yesterday.",
    "I went to the river bank to take pictures.",
    "AI is changing the future of software engineering.",
    "The software update fixed many security bugs.",
    "Cybersecurity is important for every company.",
    "Hackers attacked the company's server last night.",
    "This laptop is fast but the keyboard is uncomfortable.",
    "The new model understands the tweet context very well."
]   

df = pd.DataFrame({'tweet': tweets})
df.head(20)


In [ ]:
stop_words = set(stopwords.words('english'))

def preprocess_tweet(text):
    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'https?://\\S+|www\\.\\S+', '', text)

    # Remove @mentions
    text = re.sub(r'@\\w+', '', text)

    # Keep hashtag text but remove '#'
    text = re.sub(r'#', '', text)

    # Word tokenization using NLTK
    tokens = word_tokenize(text)

    # Keep alphabetic words and remove stop words
    tokens = [
        word for word in tokens
        if word.isalpha() and word not in stop_words
    ]

    return tokens

df['tokens'] = df['tweet'].apply(preprocess_tweet)

df[['tweet', 'tokens']]


In [ ]:
for original, tokens in zip(df['tweet'], df['tokens']):
    print('Original :', original)
    print('Cleaned  :', tokens)
    print('-' * 70)


In [ ]:
tokenized_tweets = df['tokens'].tolist()

print('Number of tweets:', len(tokenized_tweets))
print('First cleaned tweet:', tokenized_tweets[0])


In [ ]:
from gensim.models import Word2Vec

w2v_model = Word2Vec(
    sentences=tokenized_tweets,
    vector_size=100,
    window=5,
    min_count=1,
    workers=4,
    sg=1,
    epochs=100
)

word = 'bank'
vector = w2v_model.wv[word]

print('Word:', word)
print('Vector shape:', vector.shape)
print('First 10 values:', vector[:10])

print('\nSimilar words:')
print(w2v_model.wv.most_similar(word, topn=5))


In [ ]:
!pip install gensim

In [ ]:
import os
import numpy as np

def load_glove(glove_path):
    embeddings = {}

    with open(glove_path, 'r', encoding='utf-8') as f:
        for line in f:
            values = line.rstrip().split()
            word = values[0]
            vector = np.asarray(values[1:], dtype=np.float32)
            embeddings[word] = vector

    return embeddings


glove_path = '/kaggle/input/glove6b/glove.6B.100d.txt'

if os.path.exists(glove_path):
    glove = load_glove(glove_path)

    word = 'bank'

    print('Word:', word)
    print('Vector shape:', glove[word].shape)
    print('First 10 values:', glove[word][:10])

else:
    print('GloVe file not found.')

In [ ]:
df_fake.head()

In [ ]:
!pip install contractions spacy nltk -q
!python -m spacy download en_core_web_sm # spacy english model 
# contians Vocabulary
#Tokenization rules
# Part-of-speech information
# Lemmatization information
# Named entity recognition

In [ ]:
df_fake['combined_text'] = (
    df_fake['title'].fillna('') + ' ' +
    df_fake['text'].fillna('')
)

print(df_fake)

In [ ]:
import pandas as pd

pd.set_option('display.max_columns', None)
import re # regex
import nltk # tokenization, stopwords, stemming,
import spacy # lemmatization
import contractions # expands -> don't - do not

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer

nltk.download('punkt') # nltk tokenizer data
nltk.download('punkt_tab') #additional tokenizer data
nltk.download('stopwords') # the is a an etc. are the stop words

# Load spaCy model
nlp = spacy.load("en_core_web_sm")

# Stopwords
stop_words = set(stopwords.words('english'))

# Stemmer
stemmer = PorterStemmer()

In [ ]:
def clean_text(text):
    # Convert to string
    text = str(text)

    # Expand contractions
    text = contractions.fix(text)

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)

    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # Keep only alphabets and spaces
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text


df_fake['cleaned_text'] = df_fake['combined_text'].apply(clean_text)


In [ ]:
df_fake['tokens'] = df_fake['cleaned_text'].apply(word_tokenize)

print(df_fake)

In [ ]:
df_fake['filtered_tokens'] = df_fake['tokens'].apply(
    lambda tokens: [
        word for word in tokens
        if word not in stop_words
    ]
)

In [ ]:
def lemmatize_text(tokens):
    doc = nlp(" ".join(tokens))

    return [
        token.lemma_
        for token in doc
        if token.lemma_ not in stop_words
        and token.lemma_.isalpha()
    ]


df_fake['lemmatized_tokens'] = df_fake['filtered_tokens'].apply(
    lemmatize_text
)

In [ ]:
df_fake['stemmed_tokens'] = df_fake['lemmatized_tokens'].apply(
    lambda tokens: [stemmer.stem(word) for word in tokens]
)